# Spectrum Proof Point: Unified Entitlement View

Redshift Spectrum is where two entitlement systems apply simultaneously:
**Redshift RBAC** controls who can issue the SQL, and **Lake Formation**
controls who can read the underlying S3 Parquet. Both must be satisfied.

Ranger unifies both entitlement trees into a single policy view.
The auditor gets **one console instead of two.**


## Step 1: Show the two entitlement trees separately


In [ ]:
import os, logging, json
import boto3, psycopg2, requests, trino
from dotenv import load_dotenv
from IPython.display import display, Markdown, HTML

load_dotenv()
logging.basicConfig(level=logging.INFO)

# Direct Redshift connection (shows RBAC grants)
redshift_conn = psycopg2.connect(
    host=os.getenv('REDSHIFT_HOST'), port=int(os.getenv('REDSHIFT_PORT', 5439)),
    database=os.getenv('REDSHIFT_DATABASE'),
    user=os.getenv('REDSHIFT_USER'), password=os.getenv('REDSHIFT_PASSWORD'),
    sslmode='require'
)

# Lake Formation client (shows LF grants)
lf = boto3.client('lakeformation', region_name=os.getenv('AWS_REGION', 'us-east-2'))

# Ranger REST API
ranger_scheme = os.getenv('RANGER_SCHEME', 'http')
ranger_url = f"{ranger_scheme}://{os.getenv('RANGER_HOST')}:{os.getenv('RANGER_PORT', 6080)}"
ranger_auth = (os.getenv('RANGER_ADMIN_USER', 'admin'), os.getenv('RANGER_ADMIN_PASSWORD'))

display(Markdown('**Connected** to Redshift, Lake Formation, and Ranger.'))

In [ ]:
# Entitlement Tree 1: Redshift RBAC grants on the Spectrum external schema
rs_cursor = redshift_conn.cursor()
rs_cursor.execute("""
    SELECT schemaname, tablename, usename, has_table_privilege(usename, schemaname || '.' || tablename, 'SELECT') as has_select
    FROM pg_tables
    CROSS JOIN pg_user
    WHERE schemaname = 'spectrum_federation'
    AND usename IN ('admin')
    AND tablename LIKE 'ledger%'
""")
redshift_grants = rs_cursor.fetchall()
display(Markdown('### Entitlement Tree 1: Redshift RBAC (SQL execution layer)'))
if redshift_grants:
    for g in redshift_grants:
        display(Markdown(f'- Schema `{g[0]}`, table `{g[1]}`: user `{g[2]}` has SELECT = `{g[3]}`'))
else:
    # Fallback: just show the external schema exists
    rs_cursor.execute("SELECT schemaname, tablename FROM svv_external_tables WHERE schemaname = 'spectrum_federation'")
    ext_tables = rs_cursor.fetchall()
    display(Markdown(f'Spectrum external tables found: {len(ext_tables)}'))
    for t in ext_tables:
        display(Markdown(f'- `{t[0]}.{t[1]}`'))

In [ ]:
# Entitlement Tree 2: Lake Formation grants on the underlying S3/Glue data
glue_db = os.getenv('GLUE_DATABASE', 'federation_demo')
display(Markdown('### Entitlement Tree 2: Lake Formation (S3 data access layer)'))
try:
    response = lf.list_permissions(
        Resource={'Table': {'DatabaseName': glue_db, 'Name': 'ledger_entries_warm'}},
        MaxResults=100
    )
    for entry in response.get('PrincipalResourcePermissions', []):
        principal = entry['Principal']['DataLakePrincipalIdentifier']
        perms = entry['Permissions']
        display(Markdown(f'- `{principal}`: {perms}'))
except Exception as e:
    display(Markdown(f'Lake Formation query: {e}'))
    display(Markdown('*(LF grants may be at database level — checking database-level permissions)*'))
    try:
        response = lf.list_permissions(
            Resource={'Database': {'Name': glue_db}},
            MaxResults=100
        )
        for entry in response.get('PrincipalResourcePermissions', []):
            principal = entry['Principal']['DataLakePrincipalIdentifier']
            perms = entry['Permissions']
            display(Markdown(f'- `{principal}`: {perms}'))
    except Exception as e2:
        display(Markdown(f'LF database-level query: {e2}'))

## Step 2: Show the unified Ranger policy view

Both entitlement trees extracted and normalized into a single Ranger policy:


In [ ]:
# Ranger unified view: policies for Spectrum-backed tables (both Redshift + LF sources)
resp = requests.get(
    f"{ranger_url}/service/public/v2/api/policy",
    params={'serviceName': 'dev_trino'},
    auth=ranger_auth,
    timeout=30,
)
all_policies = resp.json() if resp.status_code == 200 else []

# Find policies with source labels from redshift or lake_formation
spectrum_policies = [
    p for p in all_policies
    if any(
        label.startswith('source:') for label in p.get('policyLabels', [])
    )
]

display(Markdown(f'### Ranger Unified View: {len(spectrum_policies)} extracted policies'))
display(Markdown('| Policy | Source | Type | Labels |'))
display(Markdown('|--------|--------|------|--------|'))
for p in spectrum_policies:
    labels = p.get('policyLabels', [])
    source_labels = [l for l in labels if l.startswith('source:')]
    tier_labels = [l for l in labels if l.startswith('governance_tier:')]
    ptype = {0: 'access', 1: 'masking', 2: 'row-filter'}.get(p.get('policyType', 0), 'unknown')
    display(Markdown(f'| {p["name"]} | {", ".join(source_labels)} | {ptype} | {", ".join(tier_labels)} |'))

display(Markdown(f'\n**Total:** {len(spectrum_policies)} policies from multiple sources in one console'))

## Step 3: Run a governed Spectrum query through Trino


In [ ]:
# Run a governed Spectrum query through Trino
trino_conn = trino.dbapi.connect(
    host=os.getenv('TRINO_HOST'), port=int(os.getenv('TRINO_PORT', 8080)),
    user=os.getenv('TRINO_USER', 'test_user'), catalog='redshift', schema='spectrum_federation'
)
cursor = trino_conn.cursor()

# Query the warm (Spectrum/S3) data through Trino
try:
    cursor.execute("""
        SELECT jurisdiction, count(*) as entry_count, CAST(sum(amount) AS DECIMAL(18,2)) as total_amount
        FROM redshift.spectrum_federation.ledger_entries_warm
        GROUP BY jurisdiction
        ORDER BY total_amount DESC
    """)
    rows = cursor.fetchall()
    cols = [d[0] for d in cursor.description]
    display(Markdown('### Spectrum Query Result (governed by Redshift RBAC + Lake Formation + Ranger)'))
    display(Markdown(f'| {"  |  ".join(cols)} |'))
    display(Markdown(f'| {"  |  ".join(["---"] * len(cols))} |'))
    for r in rows:
        display(Markdown(f'| {r[0]} | {r[1]:,} entries | ${float(r[2]):,.2f} |'))
except Exception as e:
    display(Markdown(f'Spectrum query through Trino: `{e}`'))
    display(Markdown('*(If Trino Hive connector is not configured for Spectrum, query directly via Redshift)*'))
    # Fallback: query Spectrum directly via Redshift
    rs_cursor.execute("""
        SELECT jurisdiction, count(*) as entry_count, sum(amount) as total_amount
        FROM spectrum_federation.ledger_entries_warm
        GROUP BY jurisdiction
        ORDER BY total_amount DESC
    """)
    rows = rs_cursor.fetchall()
    display(Markdown('### Spectrum Query (direct Redshift — dual entitlement active)'))
    for r in rows:
        display(Markdown(f'- Jurisdiction `{r[0]}`: {r[1]:,} entries, ${float(r[2]):,.2f} total'))

trino_conn.close()

## Step 4: Show unified audit record


In [ ]:
# Show Ranger audit trail — single console for both entitlement trees
display(Markdown('### Ranger Audit: Single Console'))
display(Markdown('''
Without federation, the auditor must:
1. Check Redshift RBAC grants (Redshift system tables)
2. Check Lake Formation grants (AWS Console / CloudTrail)
3. Manually correlate who accessed what, when

With federation, the auditor checks **one Ranger policy view** that shows
both entitlement trees side by side, with provenance labels tracking the source.
'''))

# Show the specific Ranger policies for spectrum data
lf_policies = [p for p in spectrum_policies if any('lake_formation' in l for l in p.get('policyLabels', []))]
rs_policies = [p for p in spectrum_policies if any('redshift' in l for l in p.get('policyLabels', []))]
display(Markdown(f'- **Lake Formation-sourced policies:** {len(lf_policies)}'))
display(Markdown(f'- **Redshift-sourced policies:** {len(rs_policies)}'))
display(Markdown(f'- **Total unified policies:** {len(spectrum_policies)}'))

redshift_conn.close()

## Key Takeaway

| Without Federation | With Federation |
|---|---|
| Auditor checks Redshift RBAC grants | Auditor checks Ranger unified view |
| Auditor checks Lake Formation grants | *(same view -- both trees visible)* |
| Auditor stitches CloudTrail + Redshift logs | Auditor reads single Ranger audit |
| Two consoles, manual correlation | **One console, one query, one answer** |
